# Experiment 2.0 — xP-excluded linear baseline

Re-run Phase 1's hurdle architecture (LR stage 1 + Ridge stage 2) with **xP removed** from both stages. This establishes the bar that the Phase 2 MLP must beat.

Two outputs gate downstream decisions:
1. **Stage 1 AUC drop (with xP vs without xP).** Phase 1 reported 0.95 with xP. If the no-xP AUC is still ≥0.93, the hurdle architecture is justified for 2.1. If it drops below ~0.90, a single-stage MLP becomes a defensible alternative and we should reconsider.
2. **Combined val R² without xP.** This is the bar 2.1 needs to beat to demonstrate that nonlinearity adds value over linear modeling in the xP-excluded setting.

Also produced: per-position MAE breakdown, Lasso coefficient comparison (with-xP vs no-xP) to verify the predicted sign flip on `total_points_roll3`.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, r2_score, mean_absolute_error

SEED = 42
np.random.seed(SEED)
pd.set_option("display.float_format", "{:.4f}".format)

## Load data and define splits

Train = 2021-22 + 2022-23. Val = 2023-24. Test (2024-25 if present) is **not** touched in this phase.


In [ ]:
DATA_PATH = Path("../data/processed/fpl_modeling_data.csv")  # adjust if notebook lives elsewhere
df = pd.read_csv(DATA_PATH)

train = df[df["season"].isin(["2021-22", "2022-23"])].copy()
val   = df[df["season"] == "2023-24"].copy()

print(f"train: {len(train):>6,} rows   val: {len(val):>6,} rows   total cols: {df.shape[1]}")

## Feature groups and `played_any` label

Drop xP from features. Derive `played_any` from the target: a non-zero `total_points` implies the player accrued match output, which strongly implies they played. The rare edge case (e.g. 1-min appearance + yellow card → 0 points) is treated as DNP. If your Phase 1 hurdle used minutes-derived `played_any`, substitute it here.

> If you maintain explicit feature-group constants from Phase 0 (e.g. `LAG_FEATURES`, `PRE_KICKOFF_FEATURES`), import them and replace the implicit list below to keep the project consistent.


In [ ]:
IDENTIFIERS = ["name", "season", "GW", "player_season", "team", "position"]
TARGET = "total_points"

all_features      = [c for c in df.columns if c not in IDENTIFIERS + [TARGET]]
features_with_xp  = all_features                    # for Stage 1 AUC sanity check + Lasso comparison
features_no_xp    = [f for f in all_features if f != "xP"]

for split in (train, val):
    split["played_any"] = (split[TARGET] != 0).astype(int)

play_rate_train = train["played_any"].mean()
play_rate_val   = val["played_any"].mean()
print(f"features (with xP): {len(features_with_xp)}   features (no xP): {len(features_no_xp)}")
print(f"play rate — train: {play_rate_train:.3f}   val: {play_rate_val:.3f}")

## Stage 1 — Logistic regression (`played_any` ~ features)

Fit two LRs: one with xP, one without. The AUC gap quantifies how much of the play/no-play signal lives in xP vs in the raw features. This is the **gating number for the 2.1 architectural decision**.

Standardization is fit on train only and applied to val. Each model gets its own scaler since the feature sets differ by one column.


In [ ]:
def fit_lr_get_auc(feat_list):
    Xtr = train[feat_list].to_numpy()
    Xva = val[feat_list].to_numpy()
    ytr = train["played_any"].to_numpy()
    yva = val["played_any"].to_numpy()

    sc = StandardScaler().fit(Xtr)
    lr = LogisticRegression(max_iter=5000, random_state=SEED).fit(sc.transform(Xtr), ytr)
    p_play = lr.predict_proba(sc.transform(Xva))[:, 1]
    return p_play, roc_auc_score(yva, p_play), sc, lr

p_play_with_xp, auc_with_xp, _, _                = fit_lr_get_auc(features_with_xp)
p_play_no_xp,   auc_no_xp,   sc_s1_no_xp, lr_s1  = fit_lr_get_auc(features_no_xp)

gap = auc_with_xp - auc_no_xp
print(f"Stage 1 AUC, WITH xP:    {auc_with_xp:.4f}   (Phase 1 reference: 0.95)")
print(f"Stage 1 AUC, WITHOUT xP: {auc_no_xp:.4f}")
print(f"AUC drop from removing xP: {gap:.4f}")
print()
if auc_no_xp >= 0.93:
    print("→ Hurdle architecture for 2.1 is justified: linear stage 1 still ~ceiling without xP.")
elif auc_no_xp >= 0.90:
    print("→ Borderline. Hurdle still defensible but worth running a single-stage MLP as sanity check.")
else:
    print("→ Reconsider hurdle. Linear stage 1 lost meaningful signal; single-stage MLP may be preferable.")

## Stage 2 — Ridge regression on played-only rows

Train Ridge on `played_any == 1` rows from the training set, no xP. `RidgeCV` over a log-spaced alpha grid handles the regularization choice. Stage 2 has its own scaler fit on the played-only training subset.


In [ ]:
played = train[train["played_any"] == 1]
Xtr_p  = played[features_no_xp].to_numpy()
ytr_p  = played[TARGET].to_numpy()
Xva    = val[features_no_xp].to_numpy()

sc_s2  = StandardScaler().fit(Xtr_p)
ridge  = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit(sc_s2.transform(Xtr_p), ytr_p)
mu_play_val = ridge.predict(sc_s2.transform(Xva))

print(f"Stage 2 RidgeCV — alpha selected: {ridge.alpha_:.4g}   (n_train_played = {len(played):,})")

## Combine and evaluate

Hurdle prediction: $\hat{y} = P(\text{play} \mid x) \cdot E[\text{points} \mid \text{played}, x]$.
No thresholding on stage 1 — multiply probabilities directly to preserve calibration on borderline rotation cases.


In [ ]:
y_pred_val = p_play_no_xp * mu_play_val
y_val      = val[TARGET].to_numpy()

r2_no_xp  = r2_score(y_val, y_pred_val)
mae_no_xp = mean_absolute_error(y_val, y_pred_val)
print(f"Hurdle (no xP) — Val R²: {r2_no_xp:.4f}   Val MAE: {mae_no_xp:.4f}")

In [ ]:
per_position = (
    val.assign(abs_err=np.abs(y_val - y_pred_val))
       .groupby("position")["abs_err"]
       .agg(MAE="mean", n="count")
       .reindex(["GK", "DEF", "MID", "FWD"])
)
per_position

## Lasso coefficient comparison

Fit a LassoCV with and without xP on the played-only subset. Two things to verify:

1. **xP dominance with xP included.** Phase 1 reported standardized coefficient ≈ 2.4 — should reproduce here.
2. **Sign flip on `total_points_roll3`.** Phase 1 found it went *negative* when xP was present (xP absorbed the matchup signal, leaving recent form as a regression-to-mean correction). Without xP, the coefficient should flip back to positive. If it doesn't, that's a real finding — investigate before moving on.


In [ ]:
def fit_lasso_coefs(feat_list):
    X = played[feat_list].to_numpy()
    y = played[TARGET].to_numpy()
    sc = StandardScaler().fit(X)
    lasso = LassoCV(cv=5, random_state=SEED, max_iter=20000, n_jobs=-1).fit(sc.transform(X), y)
    return pd.DataFrame({"feature": feat_list, "coef": lasso.coef_}), lasso.alpha_

coefs_with_xp, alpha_with = fit_lasso_coefs(features_with_xp)
coefs_no_xp,   alpha_no   = fit_lasso_coefs(features_no_xp)

top_with = coefs_with_xp.reindex(coefs_with_xp["coef"].abs().sort_values(ascending=False).index).head(10).reset_index(drop=True)
top_no   = coefs_no_xp.reindex(coefs_no_xp["coef"].abs().sort_values(ascending=False).index).head(10).reset_index(drop=True)

print(f"Lasso alpha — with xP: {alpha_with:.4g}   no xP: {alpha_no:.4g}\n")
print("Top 10 |coef|, WITH xP:");  print(top_with.to_string(index=False))
print("\nTop 10 |coef|, WITHOUT xP:");  print(top_no.to_string(index=False))

In [ ]:
key_feats = ["total_points_roll3", "total_points_lag1", "bps_roll3", "bps_lag1",
             "minutes_roll3", "minutes_lag1", "ict_index_roll3", "xP"]

sign_check = (
    coefs_with_xp.rename(columns={"coef": "coef_with_xp"})
        .merge(coefs_no_xp.rename(columns={"coef": "coef_no_xp"}), on="feature", how="left")
        .query("feature in @key_feats")
        .reset_index(drop=True)
)
sign_check["flipped"] = np.sign(sign_check["coef_with_xp"]) != np.sign(sign_check["coef_no_xp"].fillna(0))
sign_check

## Summary table

This is the row-by-row comparison the Phase 2 writeup will anchor on. The 2.1 MLP row will be added after that experiment runs.


In [ ]:
summary = pd.DataFrame([
    {"Model": "xP passthrough (Phase 1 ref)",     "Val R2": 0.5169,  "Val MAE": 0.8129},
    {"Model": "Phase 1 hurdle WITH xP (ceiling)", "Val R2": 0.6386,  "Val MAE": 0.7116},
    {"Model": "Phase 1 hurdle WITHOUT xP (2.0)",  "Val R2": r2_no_xp, "Val MAE": mae_no_xp},
    {"Model": "Hurdle MLP, xP-excluded (2.1)",    "Val R2": np.nan,   "Val MAE": np.nan},
])
summary

## What to read off this notebook before moving to 2.1

1. **Stage 1 AUC without xP** — gates the hurdle decision. Print message above gives the verdict.
2. **2.0 combined R²** — the bar 2.1 must beat. Phase 1 plan guessed 0.45–0.55. Anything materially above 0.55 weakens the "MLP recovers xP's value" framing and we should reframe before writeup.
3. **Sign of `total_points_roll3` in no-xP Lasso** — should be positive. If still negative, investigate: either Phase 1's interpretation was off or there's a feature-set mismatch.
4. **Per-position MAE** — note the worst position. Will compare against 2.1's per-position MAE to see where nonlinearity helps most.
